# Financial Research & Analytics using Agentic AI
### This is a multi-agent LangGraph pipeline which can be used for evidence-supported stock research. This is run locally in my Anaconda Navigator's Jupyter notebook via Ollama!

**Model used:** `llama3.2 is the model used since tinyllama couldn't follow complex instructions and as for the content it hallucinated more`

**Pipeline:The Pipeline is as follows**

```
Data Agent -> Market Agent -> Research Agent -> Draft Report
   -> Verification Agent -> (revise once if flagged) -> Final Report
```

| Agent | Job |
|---|---|
| Data Agent | Pulls price history + fundamentals via `yfinance` |
| Market Agent | Computes period return, volatility, moving averages, trend, index comparison — pure Python, no LLM |
| Research Agent | Pulls recent headlines via `yfinance`'s `Ticker.news`, LLM summarises them |
| Report Agent (draft) | LLM combines Market + Research into one narrative paragraph |
| Verification Agent | **Rule-based**, not LLM-judged — checks trend wording, headline dates, and claim/evidence support |
| Report Agent (revise) | Only runs if flagged; rewrites the narrative to fix the specific issues |
| Report Agent (finalize) | Assembles everything into one JSON report |

**Documented thresholds:**
- Trend: period return > +3% → uptrend, < -3% → downtrend, else sideways
- Volatility: standard deviation of daily returns, annualised (× √252)
- Moving averages: 50-day and 200-day trailing; `None` if not enough history


In [18]:
import os


## 0. Setup

Requires Ollama running locally with `llama3.2` pulled, and an internet connection for `yfinance` (Data Agent and Research Agent make real network calls to Yahoo Finance).

In [19]:

!pip install -r requirements.txt   

from state import new_state
from graph import build_graph

print("Modules loaded.")


Modules loaded.


## 1. Warm-up — Prove the Model Needs Structured Agents

We are now, asking the plain model which has no tools or agents with it a question that needs real, current
numbers it cannot know or reliably compute on its own.

In [20]:
from llm import get_llm

warmup_questions = [
    "What was Apple's exact stock return over the last quarter?",
    "What is Tesla's current volatility and 50-day moving average?",
]

for q in warmup_questions:
    response = get_llm().invoke(q)
    print("Q:", q)
    print("A:", response.content)
    print("-" * 80)


Q: What was Apple's exact stock return over the last quarter?
A: I can't provide real-time information or specific stock returns, but I can give you some general information about Apple's stock performance.

As of my knowledge cutoff in December 2023, Apple's stock had been performing well, but I don't have the exact figures for the last quarter. Apple's stock price can fluctuate rapidly and may have changed since my knowledge cutoff.

For the most up-to-date and accurate information on Apple's stock performance, I recommend checking a reliable financial news source or the company's official website. They should have the latest information on Apple's stock price and performance.
--------------------------------------------------------------------------------
Q: What is Tesla's current volatility and 50-day moving average?
A: I can’t provide real-time data or current information. However, I can offer some general information about Tesla's stock performance. Would that help?
------------

We can infer that the model is either replying with "Its performing well", "I can't provide real-time data", "Ican give you general information" etc., But...
**Why the model gets these wrong?** 
the plain model has no live market data — its knowledge is frozen at
its training cutoff and it cannot fetch today's prices or compute a real return, volatility, or moving
average. Without a Data Agent actually pulling numbers from `yfinance`, any answer here is either a
refusal, a stale guess, or a plausible-sounding fabrication.

## 2. Data Agent 
Data agent is tested directly by calling the import function. data_agent is a seperate python file.

In [21]:
from data_agent import data_agent_node

test_state = new_state("test query", "AAPL", "2024-01-01", "2024-03-31")
result = data_agent_node(test_state)
print("status:", result["data_status"])
print("price_summary:", result["price_summary"])
print("fundamentals:", result["fundamentals"])
print("close_series length:", len(result["close_series"]))




  [data_agent] AAPL: 61 trading days, 185.64 -> 171.48
status: success
price_summary: {'first_close': 185.64, 'last_close': 171.48, 'period_high': 196.38, 'period_low': 168.49, 'avg_volume': 62003215.0, 'trading_days': 61}
fundamentals: {'sector': 'Technology', 'industry': 'Consumer Electronics', 'market_cap': 4873726722048, 'trailing_pe': 38.29702, 'fifty_two_week_high': 345.34, 'fifty_two_week_low': 243.42}
close_series length: 61


In [22]:
# Edge case: an invalid ticker
bad_state = new_state("test", "NOTAREALTICKER123", "2024-01-01", "2024-03-31")
bad_result = data_agent_node(bad_state)
print("status:", bad_result["data_status"])
print("notes:", bad_result["data_notes"])


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTAREALTICKER123"}}}
$NOTAREALTICKER123: possibly delisted; no timezone found

1 Failed download:
['NOTAREALTICKER123']: possibly delisted; no timezone found


  [data_agent] no usable history for NOTAREALTICKER123 in this window
status: error
notes: No trading data for 'NOTAREALTICKER123' between 2024-01-01 and 2024-03-31. Check the ticker symbol and that the window contains trading days.


## 3. Market Agent:
Market agent, yet again, is tested directly by using python only. No networks are brought into picture

In [23]:
from market_agent import compute_market_stats

uptrend_closes = [100, 101, 103, 102, 105, 108, 110, 112, 115, 118]
print("Synthetic uptrend:", compute_market_stats(uptrend_closes, [1_000_000]*10))

downtrend_closes = [200, 198, 195, 190, 188, 182, 178, 175, 170, 165]
print("Synthetic downtrend:", compute_market_stats(downtrend_closes, [2_000_000]*10))

print("Edge case (empty):", compute_market_stats([]))


Synthetic uptrend: {'status': 'success', 'period_return_pct': 18.0, 'volatility_pct': 19.57, 'ma_short': None, 'ma_long': None, 'avg_volume': 1000000.0, 'trend': 'uptrend'}
Synthetic downtrend: {'status': 'success', 'period_return_pct': -17.5, 'volatility_pct': 13.18, 'ma_short': None, 'ma_long': None, 'avg_volume': 2000000.0, 'trend': 'downtrend'}
Edge case (empty): {'status': 'error', 'message': 'need at least 2 closing prices to compute market stats'}


In [24]:
# Now on real data, chained from the Data Agent's output
from market_agent import market_agent_node

state_with_data = data_agent_node(new_state("test", "AAPL", "2024-01-01", "2024-03-31"))
state_with_market = market_agent_node(state_with_data)
print("period_return_pct:", state_with_market["period_return_pct"])
print("volatility_pct:", state_with_market["volatility_pct"])
print("trend:", state_with_market["trend"])
print("ma_short:", state_with_market["ma_short"], " ma_long:", state_with_market["ma_long"])
print("index_comparison:", state_with_market["index_comparison"])


  [data_agent] AAPL: 61 trading days, 185.64 -> 171.48
  [market_agent] return=-7.63% vol=19.96% trend=downtrend
period_return_pct: -7.63
volatility_pct: 19.96
trend: downtrend
ma_short: 181.28  ma_long: None
index_comparison: {'benchmark': '^GSPC', 'benchmark_return_pct': 10.79, 'relative_return_pct': -18.42}


In [25]:
import yfinance as yf
raw = yf.Ticker("AAPL").news
print(type(raw), len(raw) if raw else 0)
print(raw[0] if raw else "empty")

<class 'list'> 0
empty


## 4. Research Agent — tested directly

In [26]:
from research_agent import research_agent_node

state_with_research = research_agent_node(state_with_market)
print("Headlines found:", len(state_with_research["headlines"]))
for h in state_with_research["headlines"][:5]:
    print(" -", h["date"], h["title"])
print()
print("Research summary:")
print(state_with_research["research_summary"])


  [research_agent] no headlines found for AAPL
Headlines found: 0

Research summary:
No recent news headlines were available for AAPL through the data source used. This report's narrative relies on price and volume data only.


## 5. Verification Agent — rule logic tested directly (no LLM, fully deterministic)

In [27]:
from verification_agent import verify

headlines_sample = [
    {"title": "Company reports strong quarterly earnings beat", "date": "2024-03-05", "publisher": "Reuters"},
]

print("Consistent narrative (expect VERIFIED):")
print(verify(
    "The stock showed a clear uptrend, supported by strong quarterly earnings.",
    "uptrend", headlines_sample, "2024-03-01", "2024-03-31"
))
print()

print("Contradictory trend words (expect FLAGGED - trend_word_check):")
print(verify(
    "The stock declined sharply this period.",
    "uptrend", headlines_sample, "2024-03-01", "2024-03-31"
))
print()

print("Unsupported claim (expect FLAGGED - keyword_support_check):")
print(verify(
    "The stock rallied following a major merger announcement.",
    "uptrend", headlines_sample, "2024-03-01", "2024-03-31"
))
print()

print("Headline outside window (expect FLAGGED - date_window_check):")
old_headlines = headlines_sample + [{"title": "Old story", "date": "2023-01-01", "publisher": "X"}]
print(verify(
    "The stock rallied this period on strong earnings.",
    "uptrend", old_headlines, "2024-03-01", "2024-03-31"
))


Consistent narrative (expect VERIFIED):
{'status': 'verified', 'flags': []}

Contradictory trend words (expect FLAGGED - trend_word_check):
{'status': 'flagged', 'flags': [{'rule': 'trend_word_check', 'detail': "Narrative uses downward language (['declined']) but classified trend is 'uptrend'.", 'severity': 'high'}]}

Unsupported claim (expect FLAGGED - keyword_support_check):
{'status': 'flagged', 'flags': [{'rule': 'keyword_support_check', 'detail': "Narrative references 'merger/acquisition' but no retrieved headline mentions it.", 'severity': 'high'}]}

Headline outside window (expect FLAGGED - date_window_check):
{'status': 'flagged', 'flags': [{'rule': 'date_window_check', 'detail': "Headline 'Old story' dated 2023-01-01 falls outside 2024-03-01 to 2024-03-31.", 'severity': 'low'}]}


## 6. Report Agent — draft narrative tested directly

In [28]:
from report_agent import draft_report_node

state_with_draft = draft_report_node(state_with_research)
print("Draft narrative:")
print(state_with_draft["draft_narrative"])


  [report_agent:draft] narrative drafted
Draft narrative:
Over the specified period from January 1st, 2024, to March 31st, 2024, Apple Inc.'s (AAPL) stock has been in a downtrend, as evidenced by its -7.63% return. This decline is accompanied by an increase in volatility, with an annualized volatility of 19.96%. The stock's price has been influenced by its 50-day moving average, which currently stands at 181.28, suggesting a resistance level. The lack of recent news headlines for AAPL limits our ability to identify specific drivers of the trend, but the data suggests a continuation of the downtrend. The average daily volume of 62.03 billion shares indicates a relatively high level of trading activity, which may contribute to the stock's price movements.


## 7. Full Graph — Demo Scenarios

Three scenarios run through the complete compiled graph:

1. A recent 3-month window for a large, liquid stock (direction will be whatever the live data shows)
2. A different ticker/window, to see a different trend classification
3. A **verification stress test** — a deliberately false narrative is injected directly into the
   Verification Agent (bypassing the LLM draft) to prove the rule-based checks catch it reliably,
   independent of whatever mood the live LLM happens to be in that day.

In [29]:
from graph import build_graph

graph = build_graph()

def run_scenario(label, ticker, start_date, end_date):
    print("=" * 78)
    print(label)
    print("=" * 78)
    query = f"How has {ticker} performed between {start_date} and {end_date}?"
    result = graph.invoke(new_state(query, ticker, start_date, end_date))

    print()
    print("Ticker:", result["ticker"])
    print("Trend:", result["trend"], " | Period return:", result["period_return_pct"], "%")
    print("Verification status:", result["verification_status"], "| revisions:", result["revision_count"])
    print()
    print("Final narrative:")
    print(result["final_narrative"])
    print()
    return result

result_1 = run_scenario("SCENARIO 1", "AAPL", "2024-01-01", "2024-03-31")


SCENARIO 1
  [data_agent] AAPL: 61 trading days, 185.64 -> 171.48
  [market_agent] return=-7.63% vol=19.96% trend=downtrend
  [research_agent] no headlines found for AAPL
  [report_agent:draft] narrative drafted
  [verification_agent] no issues found — narrative verified
  [finalize_report] report assembled

Ticker: AAPL
Trend: downtrend  | Period return: -7.63 %
Verification status: verified | revisions: 0

Final narrative:
Over the specified period from 2024-01-01 to 2024-03-31, Apple Inc.'s (AAPL) stock price has been trending downwards, classified as a downtrend. The stock's price return of -7.63% over the period indicates a decline in value. The 50-day moving average of 181.28 provides further support for this downtrend, as it is above the current price, suggesting that the stock is currently trading below its short-term average. The high average daily volume of 62,003,215.0 further emphasizes the liquidity of the stock during this period. The downtrend is consistent with the over

In [30]:
result_2 = run_scenario("SCENARIO 2", "TSLA", "2024-01-01", "2024-06-30")


SCENARIO 2
  [data_agent] TSLA: 124 trading days, 248.42 -> 197.88
  [market_agent] return=-20.34% vol=53.09% trend=downtrend
  [research_agent] no headlines found for TSLA
  [report_agent:draft] narrative drafted
  [verification_agent] no issues found — narrative verified
  [finalize_report] report assembled

Ticker: TSLA
Trend: downtrend  | Period return: -20.34 %
Verification status: verified | revisions: 0

Final narrative:
Over the period from 2024-01-01 to 2024-06-30, Tesla Inc. (TSLA) has experienced a downtrend, with its stock price declining by 20.34% from the starting point. The 50-day moving average of $176.46 has been a significant resistance level, as the stock price has consistently traded below this level, indicating a bearish bias. The high average daily volume of 95308752.0 has contributed to the stock's volatility, with an annualized volatility of 53.09%. The lack of a 200-day moving average due to insufficient history does not provide additional context, but the over

### Scenario 3 — Verification stress test

This does not run the full graph. It directly hands the Verification Agent a
narrative containing a **deliberately false claim** (a fabricated merger that
is not in the retrieved headlines) plus a real trend/word contradiction, to
prove the rule-based checks fire reliably. This is more reproducible for
grading purposes than hoping the live LLM hallucinates on demand.

In [31]:
from verification_agent import verify

fake_narrative = (
    "The stock surged upward this period on news of a major merger deal, "
    "even though the broader numbers actually declined."
)
real_headlines = result_1["headlines"]  # whatever Scenario 1 actually retrieved

stress_result = verify(
    narrative=fake_narrative,
    trend="downtrend",   # deliberately mismatched against the "surged" language above
    headlines=real_headlines,
    start_date=result_1["start_date"],
    end_date=result_1["end_date"],
)

print("Injected narrative:", fake_narrative)
print()
print("Verification result:")
for f in stress_result["flags"]:
    print(f" - ({f['severity']}) {f['rule']}: {f['detail']}")
print()
print("Status:", stress_result["status"], f"-- {len(stress_result['flags'])} flag(s) raised")


Injected narrative: The stock surged upward this period on news of a major merger deal, even though the broader numbers actually declined.

Verification result:
 - (high) trend_word_check: Narrative uses upward language (['surged']) but classified trend is 'downtrend'.
 - (high) keyword_support_check: Narrative references 'merger/acquisition' but no retrieved headline mentions it.

Status: flagged -- 2 flag(s) raised


## 8. Save Reports

In [32]:
from report import save_report

save_report(result_1, "reports/session_report_1.json")
save_report(result_2, "reports/session_report_2.json")
print("Both reports saved to the reports/ folder.")


  [report] saved to reports/session_report_1.json
  [report] saved to reports/session_report_2.json
Both reports saved to the reports/ folder.


## 9. Reflection

*(Fill this in after running on your machine — what did the live data actually
show, did the LLM's draft narrative ever get flagged by Verification on a
real run, and did the revise step produce a better paragraph?)*

> Scenario 1 (`[ticker]`, `[dates]`) classified as **[trend]** with a period return
> of **[X]%**. [Was it flagged? If so, what did the revise step change?]
>
> Scenario 2 (`[ticker]`, `[dates]`) classified as **[trend]**... [same analysis]
>
> The verification stress test in Scenario 3 confirmed the rule-based checks
> fire correctly on a deliberately fabricated claim, independent of the live
> LLM's mood that day — this was the main point of keeping verification
> rule-based rather than LLM-judged, since an LLM asked to verify its own
> output has an unavoidable circularity problem that a fixed rule set does not.


In [33]:
import sys
!{sys.executable} -m pip install langchain langchain-core langchain-ollama langgraph yfinance

In [34]:
import sys
!{sys.executable} -m pip install langchain langchain-core langchain-ollama langgraph yfinance